<a href="https://colab.research.google.com/github/EnzoAA004/PFI_MVPTest_Enzo_AImodule/blob/enzo%2Fp10-8-clinical-expansion-preflight/76_P10_8_T1_T2_and_sagittal_axial_alignment_audit_FAST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 76 — P10.8: auditoría T1/T2 y alineación sagital–axial

Este notebook audita dos requisitos separados:

1. **Disponibilidad T1/T2 dentro de una misma cohorte**, sin mezclar pacientes de
   datasets diferentes.
2. **Viabilidad geométrica de la correspondencia sagital–axial**, usando metadatos
   DICOM cuando están disponibles (`StudyInstanceUID`, `SeriesInstanceUID`,
   `FrameOfReferenceUID`, `ImagePositionPatient`, `ImageOrientationPatient`).

El objetivo es determinar si existe evidencia suficiente para una futura
correspondencia multiplanar automática. No construye dicha automatización.

Reglas:

- no entrena;
- no carga ni deserializa `.pt`;
- no abre tests sellados;
- no lee pixel data DICOM;
- no exporta PatientID ni UIDs DICOM en claro;
- no empareja cohortes diferentes;
- no crea ground truth clínico;
- no habilita entrenamiento;
- no constituye diagnóstico clínico.


**Implementación FAST:** para evitar miles de lecturas remotas desde Google Drive, la auditoría DICOM usa un header representativo por carpeta de serie. Esto permite el preflight de plano/T1-T2/FrameOfReference/geometría sin afirmar validación completa de todos los cortes.

In [1]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Entorno no Colab")


Mounted at /content/drive


In [2]:
# pydicom se usa únicamente para leer headers DICOM con stop_before_pixels=True.
try:
    import pydicom
except ImportError:
    import subprocess, sys
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "pydicom"]
    )
    import pydicom

print("pydicom:", pydicom.__version__)


pydicom: 3.0.2


In [3]:
from __future__ import annotations

import hashlib
import json
import math
import os
import re
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import pydicom

ROOT = Path(
    os.getenv(
        "PFI_ROOT",
        "/content/drive/MyDrive/PFI_MVP",
    )
)
PREF = Path(
    os.getenv(
        "PFI_P10_8_PREFLIGHT_ROOT",
        str(
            ROOT
            / "results"
            / "P10_8_clinical_expansion_preflight"
        ),
    )
)
N75_ROOT = Path(
    os.getenv(
        "PFI_P10_8_NOTEBOOK75_ROOT",
        str(PREF / "multiframe_hernia_review"),
    )
)
OUT = Path(
    os.getenv(
        "PFI_P10_8_NOTEBOOK76_ROOT",
        str(PREF / "t1_t2_sagittal_axial_alignment"),
    )
)

MARKER_75 = N75_ROOT / "NOTEBOOK_75_COMPLETE.json"
SOURCE_READINESS_75 = (
    N75_ROOT / "multiframe_source_readiness_v1.csv"
)
PAIRING_75 = (
    N75_ROOT / "cross_plane_pairing_assessment_v1.csv"
)

for required in (
    MARKER_75,
    SOURCE_READINESS_75,
    PAIRING_75,
):
    if not required.is_file():
        raise FileNotFoundError(
            f"Falta entrada requerida: {required}"
        )

marker_75 = json.loads(
    MARKER_75.read_text(encoding="utf-8")
)

if marker_75.get("status") != "NOTEBOOK_75_COMPLETE":
    raise RuntimeError("Notebook 75 no está cerrado.")

if marker_75.get("schemaVersion") != (
    "pfi.p10-8.notebook-75-complete.v1"
):
    raise RuntimeError(
        "Notebook 75 no tiene el schemaVersion esperado."
    )

for field in (
    "trainingExecuted",
    "retrainingAuthorized",
    "trainingAuthorized",
    "weightsDeserialized",
):
    if marker_75.get(field) is not False:
        raise RuntimeError(
            f"Notebook 75 no mantiene {field}=false"
        )

if marker_75.get("outputPtFileCount") != 0:
    raise RuntimeError(
        "Notebook 75 reporta archivos .pt en salida."
    )

readiness_75 = pd.read_csv(SOURCE_READINESS_75)
pairing_75 = pd.read_csv(PAIRING_75)

truthy_pairing = (
    pairing_75["pairingAllowed"]
    .astype(str)
    .str.lower()
    .isin({"true", "1", "yes"})
)

if truthy_pairing.any():
    raise RuntimeError(
        "Notebook 75 contiene un pairing cross-cohort "
        "inesperadamente habilitado."
    )

print("Notebook 75 verificado.")
print("Salida Notebook 76:", OUT)


Notebook 75 verificado.
Salida Notebook 76: /content/drive/MyDrive/PFI_MVP/results/P10_8_clinical_expansion_preflight/t1_t2_sagittal_axial_alignment


In [4]:
def write_json(path: Path, payload: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(
            payload,
            indent=2,
            ensure_ascii=False,
            sort_keys=True,
        )
        + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)

def sha_text(value: Any) -> str | None:
    if value is None:
        return None
    text = str(value).strip()
    if not text:
        return None
    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(block)
    return digest.hexdigest()

def clean_text(value: Any) -> str:
    if value is None:
        return ""
    return str(value).strip()

def finite_vector(
    value: Any,
    expected_length: int,
) -> list[float] | None:
    if value is None:
        return None
    try:
        values = [float(x) for x in value]
    except Exception:
        return None
    if len(values) != expected_length:
        return None
    if not all(math.isfinite(x) for x in values):
        return None
    return values

def classify_plane(
    orientation: list[float] | None,
) -> str:
    if orientation is None or len(orientation) != 6:
        return "unknown"

    row = np.asarray(orientation[:3], dtype=float)
    col = np.asarray(orientation[3:], dtype=float)
    normal = np.cross(row, col)

    norm = float(np.linalg.norm(normal))
    if norm <= 1e-8:
        return "invalid"

    normal = np.abs(normal / norm)
    axis = int(np.argmax(normal))
    strength = float(normal[axis])

    # Se exige una componente dominante clara.
    if strength < 0.85:
        return "oblique"

    return {
        0: "sagittal",
        1: "coronal",
        2: "axial",
    }[axis]

T1_PATTERN = re.compile(
    r"(^|[^a-z0-9])t1([^a-z0-9]|$)",
    re.IGNORECASE,
)
T2_PATTERN = re.compile(
    r"(^|[^a-z0-9])t2([^a-z0-9]|$)",
    re.IGNORECASE,
)

def classify_weighting(*values: Any) -> str:
    text = " ".join(clean_text(v) for v in values)

    t1 = bool(T1_PATTERN.search(text))
    t2 = bool(T2_PATTERN.search(text))

    if t1 and not t2:
        return "T1"
    if t2 and not t1:
        return "T2"
    if t1 and t2:
        return "ambiguous_T1_T2"
    return "unknown"

input_hashes = {
    "NOTEBOOK_75_COMPLETE.json":
        sha256_file(MARKER_75),
    "multiframe_source_readiness_v1.csv":
        sha256_file(SOURCE_READINESS_75),
    "cross_plane_pairing_assessment_v1.csv":
        sha256_file(PAIRING_75),
}

display(
    pd.DataFrame(
        [
            {"inputName": k, "sha256": v}
            for k, v in input_hashes.items()
        ]
    )
)


,inputName,sha256
0,NOTEBOOK_75_COMPLETE.json,9668e7d944041715603a1d5f62a11ed336bfce2000b0ae...
1,multiframe_source_readiness_v1.csv,6c7c8984e0f823a93416dad04bd3b91bf16323912a3baf...
2,cross_plane_pairing_assessment_v1.csv,8e2b866c804aa2f1ca4b9abb61d6e8684b2b766038eeb6...


## A. Auditoría DICOM Al‑Kafri

Se leen únicamente headers de archivos `.ima`, con `stop_before_pixels=True`.
Los identificadores de paciente, estudio, serie y frame of reference se exportan solo
como hashes SHA-256.

La clasificación T1/T2 utiliza únicamente texto de `SeriesDescription`,
`ProtocolName`, `SequenceName` y nombre de carpeta. Si la evidencia es ambigua,
queda como `unknown` o `ambiguous_T1_T2`.


In [5]:
ALK_ROOT = Path(
    os.getenv(
        "PFI_ALKAFRI_ROOT",
        str(
            ROOT
            / "data"
            / "AXIAL_ALKAFRI"
            / "extracted"
            / "_nested"
        ),
    )
)
MRI_ROOT = (
    ALK_ROOT
    / "main_dataset__MRI_Data"
    / "01_MRI_Data"
)

if not MRI_ROOT.is_dir():
    raise FileNotFoundError(
        f"No existe MRI_ROOT Al-Kafri: {MRI_ROOT}"
    )

# FAST audit:
# - enumeramos los .ima una sola vez;
# - agrupamos por carpeta de serie;
# - leemos UN header representativo por serie.
# Esto es suficiente para preflight de plano, weighting, UID de estudio/serie/FOR
# y presencia de geometría, sin afirmar que todos los cortes de la serie ya
# fueron validados.
MAX_SERIES = int(
    os.getenv(
        "PFI_P10_8_MAX_DICOM_SERIES_76",
        "5000",
    )
)

all_ima_paths = list(MRI_ROOT.rglob("*.ima"))
series_files = {}

for path in all_ima_paths:
    series_files.setdefault(path.parent, []).append(path)

series_dirs = sorted(
    series_files,
    key=lambda p: sha_text(str(p)) or "",
)[:MAX_SERIES]

dicom_paths = []

for series_dir in series_dirs:
    files = sorted(series_files[series_dir])
    if files:
        dicom_paths.append(files[0])

series_file_count_by_folder_hash = {
    sha_text(str(series_dir.relative_to(MRI_ROOT))):
        len(series_files[series_dir])
    for series_dir in series_dirs
}

print("Archivos .ima encontrados:", len(all_ima_paths))
print("Carpetas de serie encontradas:", len(series_files))
print("Series a auditar:", len(series_dirs))
print("Headers representativos a leer:", len(dicom_paths))
print("Modo: 1 header representativo por carpeta de serie")

Archivos .ima encontrados: 17497
Carpetas de serie encontradas: 1363
Series a auditar: 1363
Headers representativos a leer: 1363
Modo: 1 header representativo por carpeta de serie


In [6]:
import time

dicom_rows = []
started = time.time()

tags = [
    "PatientID",
    "StudyInstanceUID",
    "SeriesInstanceUID",
    "FrameOfReferenceUID",
    "StudyDescription",
    "SeriesDescription",
    "ProtocolName",
    "SequenceName",
    "ImagePositionPatient",
    "ImageOrientationPatient",
    "PixelSpacing",
    "SliceThickness",
    "SpacingBetweenSlices",
    "InstanceNumber",
    "Rows",
    "Columns",
]

for index, path in enumerate(dicom_paths, start=1):
    folder_hash = sha_text(
        str(path.parent.relative_to(MRI_ROOT))
    )

    row = {
        "sourceFamily": "ALKAFRI",
        "pathHash": sha_text(
            str(path.relative_to(MRI_ROOT))
        ),
        "seriesFolderHash": folder_hash,
        "seriesFileCountOnDisk":
            series_file_count_by_folder_hash.get(folder_hash),
        "representativeHeaderOnly": True,
        "readSucceeded": False,
        "readErrorType": None,
    }

    try:
        ds = pydicom.dcmread(
            str(path),
            stop_before_pixels=True,
            force=True,
            specific_tags=tags,
        )

        orientation = finite_vector(
            getattr(
                ds,
                "ImageOrientationPatient",
                None,
            ),
            6,
        )
        position = finite_vector(
            getattr(
                ds,
                "ImagePositionPatient",
                None,
            ),
            3,
        )
        spacing = finite_vector(
            getattr(ds, "PixelSpacing", None),
            2,
        )

        series_description = clean_text(
            getattr(ds, "SeriesDescription", "")
        )
        protocol_name = clean_text(
            getattr(ds, "ProtocolName", "")
        )
        sequence_name = clean_text(
            getattr(ds, "SequenceName", "")
        )

        weighting = classify_weighting(
            series_description,
            protocol_name,
            sequence_name,
            path.parent.name,
        )

        row.update(
            {
                "patientHash":
                    sha_text(
                        getattr(ds, "PatientID", None)
                    ),
                "studyUidHash":
                    sha_text(
                        getattr(
                            ds,
                            "StudyInstanceUID",
                            None,
                        )
                    ),
                "seriesUidHash":
                    sha_text(
                        getattr(
                            ds,
                            "SeriesInstanceUID",
                            None,
                        )
                    ),
                "frameOfReferenceUidHash":
                    sha_text(
                        getattr(
                            ds,
                            "FrameOfReferenceUID",
                            None,
                        )
                    ),
                "plane":
                    classify_plane(orientation),
                "weighting": weighting,
                "orientationPresent":
                    orientation is not None,
                "positionPresent":
                    position is not None,
                "pixelSpacingPresent":
                    spacing is not None,
                "geometryComplete":
                    bool(
                        orientation is not None
                        and position is not None
                        and spacing is not None
                    ),
                "rows":
                    int(ds.Rows)
                    if getattr(ds, "Rows", None)
                    is not None
                    else None,
                "columns":
                    int(ds.Columns)
                    if getattr(ds, "Columns", None)
                    is not None
                    else None,
                "instanceNumber":
                    int(ds.InstanceNumber)
                    if str(
                        getattr(
                            ds,
                            "InstanceNumber",
                            "",
                        )
                    ).strip().isdigit()
                    else None,
                "seriesTextHash":
                    sha_text(
                        "|".join(
                            [
                                series_description,
                                protocol_name,
                                sequence_name,
                            ]
                        )
                    ),
                "readSucceeded": True,
                "readErrorType": None,
            }
        )

    except Exception as exc:
        row["readErrorType"] = type(exc).__name__

    dicom_rows.append(row)

    if index % 100 == 0 or index == len(dicom_paths):
        elapsed = time.time() - started
        print(
            f"{index}/{len(dicom_paths)} headers "
            f"({elapsed:.1f}s)"
        )

dicom_headers = pd.DataFrame(dicom_rows)

print(
    "Headers representativos leídos correctamente:",
    int(dicom_headers["readSucceeded"].sum()),
    "/",
    len(dicom_headers),
)

display(
    dicom_headers[
        dicom_headers["readSucceeded"]
    ]
    .groupby(
        [
            "plane",
            "weighting",
            "geometryComplete",
        ],
        dropna=False,
    )
    .size()
    .reset_index(name="seriesRepresentativeCount")
)

100/1363 headers (35.2s)
200/1363 headers (70.8s)
300/1363 headers (106.8s)
400/1363 headers (140.4s)
500/1363 headers (175.2s)
600/1363 headers (207.4s)
700/1363 headers (240.0s)
800/1363 headers (273.8s)
900/1363 headers (306.0s)
1000/1363 headers (338.7s)
1100/1363 headers (372.1s)
1200/1363 headers (406.4s)
1300/1363 headers (441.3s)
1363/1363 headers (461.8s)
Headers representativos leídos correctamente: 1363 / 1363


,plane,weighting,geometryComplete,seriesRepresentativeCount
0,axial,T1,True,206
1,axial,T2,True,223
2,axial,unknown,True,4
3,coronal,T1,True,18
4,coronal,T2,True,95
5,coronal,unknown,True,8
6,oblique,T2,True,2
7,sagittal,T1,True,229
8,sagittal,T2,True,326
9,sagittal,unknown,True,252


In [7]:
# Registro por serie usando un header representativo por carpeta.
# No se afirma que todos los cortes de una serie fueron auditados.
valid_headers = dicom_headers[
    dicom_headers["readSucceeded"]
].copy()

series_registry = (
    valid_headers.groupby(
        [
            "patientHash",
            "studyUidHash",
            "seriesUidHash",
            "frameOfReferenceUidHash",
            "seriesFolderHash",
            "plane",
            "weighting",
        ],
        dropna=False,
    )
    .agg(
        representativeHeaderCount=("pathHash", "count"),
        seriesFileCountOnDisk=("seriesFileCountOnDisk", "max"),
        representativeGeometryCompleteCount=(
            "geometryComplete",
            "sum",
        ),
        representativePositionPresentCount=(
            "positionPresent",
            "sum",
        ),
        representativeOrientationPresentCount=(
            "orientationPresent",
            "sum",
        ),
        representativePixelSpacingPresentCount=(
            "pixelSpacingPresent",
            "sum",
        ),
    )
    .reset_index()
)

series_registry[
    "representativeGeometryComplete"
] = (
    series_registry["representativeGeometryCompleteCount"]
    == series_registry["representativeHeaderCount"]
)

series_registry["fullSeriesGeometryValidated"] = False

display(
    series_registry.groupby(
        [
            "plane",
            "weighting",
            "representativeGeometryComplete",
        ],
        dropna=False,
    )
    .size()
    .reset_index(name="seriesCount")
)

,plane,weighting,representativeGeometryComplete,seriesCount
0,axial,T1,True,206
1,axial,T2,True,223
2,axial,unknown,True,4
3,coronal,T1,True,18
4,coronal,T2,True,95
5,coronal,unknown,True,8
6,oblique,T2,True,2
7,sagittal,T1,True,229
8,sagittal,T2,True,326
9,sagittal,unknown,True,252


In [8]:
# T1/T2 dentro de la misma cohorte Al-Kafri.
# La clave es paciente+estudio hash; nunca se compara contra SPIDER.
study_groups = []

for (patient_hash, study_hash), group in (
    series_registry.groupby(
        ["patientHash", "studyUidHash"],
        dropna=False,
    )
):
    weights = set(
        group["weighting"]
        .dropna()
        .astype(str)
    )
    planes = set(
        group["plane"]
        .dropna()
        .astype(str)
    )

    study_groups.append(
        {
            "patientHash": patient_hash,
            "studyUidHash": study_hash,
            "hasT1": "T1" in weights,
            "hasT2": "T2" in weights,
            "hasBothT1T2":
                "T1" in weights and "T2" in weights,
            "hasSagittal": "sagittal" in planes,
            "hasAxial": "axial" in planes,
            "hasBothSagittalAxial":
                "sagittal" in planes
                and "axial" in planes,
            "seriesCount": int(len(group)),
        }
    )

alkafri_study_readiness = pd.DataFrame(
    study_groups
)

display(
    pd.DataFrame(
        [
            {
                "cohort": "ALKAFRI",
                "studyCount":
                    int(len(alkafri_study_readiness)),
                "studiesWithT1":
                    int(
                        alkafri_study_readiness[
                            "hasT1"
                        ].sum()
                    ),
                "studiesWithT2":
                    int(
                        alkafri_study_readiness[
                            "hasT2"
                        ].sum()
                    ),
                "studiesWithBothT1T2":
                    int(
                        alkafri_study_readiness[
                            "hasBothT1T2"
                        ].sum()
                    ),
                "studiesWithSagittal":
                    int(
                        alkafri_study_readiness[
                            "hasSagittal"
                        ].sum()
                    ),
                "studiesWithAxial":
                    int(
                        alkafri_study_readiness[
                            "hasAxial"
                        ].sum()
                    ),
                "studiesWithBothSagittalAxial":
                    int(
                        alkafri_study_readiness[
                            "hasBothSagittalAxial"
                        ].sum()
                    ),
            }
        ]
    )
)


,cohort,studyCount,studiesWithT1,studiesWithT2,studiesWithBothT1T2,studiesWithSagittal,studiesWithAxial,studiesWithBothSagittalAxial
0,ALKAFRI,204,195,199,195,204,198,198


## B. Auditoría T1/T2 de SPIDER/P10.7

Para P10.7 se usan únicamente manifests ya generados. Esta sección no abre checkpoints
ni datasets sellados.

El objetivo es comprobar si existe evidencia tabular de disponibilidad T1/T2 por
paciente/nivel. No se intenta derivar geometría DICOM si el manifest no la contiene.


In [9]:
P10_7_ROOT = Path(
    os.getenv(
        "PFI_P10_7_RESULTS_ROOT",
        str(
            ROOT
            / "results"
            / "P10_7_spider_degenerative"
        ),
    )
)

manifest_candidates = [
    P10_7_ROOT
    / "disc_level_manifest_v1.csv",
]

# Búsqueda limitada dentro de la carpeta P10.7.
if P10_7_ROOT.is_dir():
    manifest_candidates.extend(
        P10_7_ROOT.rglob(
            "disc_level_manifest_v1.csv"
        )
    )

manifest_candidates = [
    p
    for p in dict.fromkeys(
        Path(p) for p in manifest_candidates
    )
    if p.is_file()
]

if not manifest_candidates:
    raise FileNotFoundError(
        "No se encontró disc_level_manifest_v1.csv "
        "en P10.7."
    )

SPIDER_MANIFEST = manifest_candidates[0]
spider = pd.read_csv(
    SPIDER_MANIFEST,
    dtype=str,
    low_memory=False,
)

print("SPIDER manifest:", SPIDER_MANIFEST)
print("Filas:", len(spider))
print("Columnas:", list(spider.columns))


SPIDER manifest: /content/drive/MyDrive/PFI_MVP/results/P10_7_spider_degenerative/disc_level_manifest_v1.csv
Filas: 1518
Columnas: ['sample_id', 'Patient', 'ivd_label', 'raw_disc_label', 'split', 'crop_path', 't1_available', 't2_available', 'crop_metadata_json', 'pfirrmann_grade', 'modic_change', 'upper_endplate_change', 'lower_endplate_change', 'spondylolisthesis', 'disc_herniation', 'disc_narrowing', 'disc_bulging']


In [10]:
# Se detectan columnas de disponibilidad/modalidad de forma conservadora.
column_names = [str(c) for c in spider.columns]

patient_candidates = [
    c
    for c in column_names
    if re.fullmatch(
        r"(patient|patient_id|case|case_id)",
        c,
        re.IGNORECASE,
    )
]
level_candidates = [
    c
    for c in column_names
    if re.search(
        r"(disc.*level|level|ivd)",
        c,
        re.IGNORECASE,
    )
]

patient_col = (
    patient_candidates[0]
    if patient_candidates
    else None
)
level_col = (
    level_candidates[0]
    if level_candidates
    else None
)

t1_cols = [
    c
    for c in column_names
    if re.search(r"t1", c, re.IGNORECASE)
]
t2_cols = [
    c
    for c in column_names
    if re.search(r"t2", c, re.IGNORECASE)
]

print("patient_col:", patient_col)
print("level_col:", level_col)
print("T1 columns:", t1_cols)
print("T2 columns:", t2_cols)

def row_has_nonempty(
    frame: pd.DataFrame,
    columns: list[str],
) -> pd.Series:
    if not columns:
        return pd.Series(
            False,
            index=frame.index,
        )
    values = (
        frame[columns]
        .fillna("")
        .astype(str)
        .apply(
            lambda col:
                col.str.strip().ne("")
        )
    )
    return values.any(axis=1)

spider_readiness = spider.copy()
spider_readiness["_hasT1"] = row_has_nonempty(
    spider_readiness,
    t1_cols,
)
spider_readiness["_hasT2"] = row_has_nonempty(
    spider_readiness,
    t2_cols,
)
spider_readiness["_hasBothT1T2"] = (
    spider_readiness["_hasT1"]
    & spider_readiness["_hasT2"]
)

spider_summary = {
    "rowCount": int(len(spider_readiness)),
    "patientColumnResolved":
        patient_col is not None,
    "levelColumnResolved":
        level_col is not None,
    "t1EvidenceColumnCount": len(t1_cols),
    "t2EvidenceColumnCount": len(t2_cols),
    "rowsWithT1":
        int(spider_readiness["_hasT1"].sum()),
    "rowsWithT2":
        int(spider_readiness["_hasT2"].sum()),
    "rowsWithBothT1T2":
        int(
            spider_readiness[
                "_hasBothT1T2"
            ].sum()
        ),
}

if patient_col is not None:
    spider_summary["uniquePatientCount"] = int(
        spider[patient_col]
        .dropna()
        .astype(str)
        .nunique()
    )
else:
    spider_summary["uniquePatientCount"] = None

print(json.dumps(spider_summary, indent=2))


patient_col: Patient
level_col: ivd_label
T1 columns: ['t1_available']
T2 columns: ['t2_available']
{
  "rowCount": 1518,
  "patientColumnResolved": true,
  "levelColumnResolved": true,
  "t1EvidenceColumnCount": 1,
  "t2EvidenceColumnCount": 1,
  "rowsWithT1": 1518,
  "rowsWithT2": 1518,
  "rowsWithBothT1T2": 1518,
  "uniquePatientCount": 218
}


## C. Gate geométrico sagital–axial dentro de una misma cohorte

Una futura correspondencia automática sagital↔axial solo puede considerarse
geométricamente auditable cuando existe, para el **mismo paciente/estudio**:

- al menos una serie sagital;
- al menos una serie axial;
- `FrameOfReferenceUID` compatible;
- `ImageOrientationPatient`;
- `ImagePositionPatient`;
- calibración física suficiente.

Este notebook no acepta como evidencia de pairing una coincidencia de `PatientID`,
número de caso o nivel entre SPIDER y Al-Kafri.


In [11]:
# Candidatos geométricos exclusivamente dentro de Al-Kafri.
# IMPORTANTE: esta es una preselección de series basada en un header
# representativo por serie. No valida todavía todos los cortes.
geometry_pairs = []

for (patient_hash, study_hash), group in (
    series_registry.groupby(
        ["patientHash", "studyUidHash"],
        dropna=False,
    )
):
    sagittal = group[
        group["plane"] == "sagittal"
    ]
    axial = group[
        group["plane"] == "axial"
    ]

    if sagittal.empty or axial.empty:
        continue

    for _, sag in sagittal.iterrows():
        for _, axi in axial.iterrows():
            same_for = bool(
                pd.notna(
                    sag["frameOfReferenceUidHash"]
                )
                and pd.notna(
                    axi["frameOfReferenceUidHash"]
                )
                and sag["frameOfReferenceUidHash"]
                == axi["frameOfReferenceUidHash"]
            )

            representative_geometry_ok = bool(
                sag["representativeGeometryComplete"]
                and axi["representativeGeometryComplete"]
            )

            geometry_pairs.append(
                {
                    "sourceFamily": "ALKAFRI",
                    "patientHash": patient_hash,
                    "studyUidHash": study_hash,
                    "sagittalSeriesUidHash":
                        sag["seriesUidHash"],
                    "axialSeriesUidHash":
                        axi["seriesUidHash"],
                    "sameFrameOfReference":
                        same_for,
                    "sagittalRepresentativeGeometryComplete":
                        bool(
                            sag["representativeGeometryComplete"]
                        ),
                    "axialRepresentativeGeometryComplete":
                        bool(
                            axi["representativeGeometryComplete"]
                        ),
                    "geometryPairingCandidate":
                        bool(
                            same_for
                            and representative_geometry_ok
                        ),
                    "fullSeriesGeometryValidated": False,
                    "automaticCrossReferenceValidated":
                        False,
                }
            )

geometry_pairing = pd.DataFrame(
    geometry_pairs,
    columns=[
        "sourceFamily",
        "patientHash",
        "studyUidHash",
        "sagittalSeriesUidHash",
        "axialSeriesUidHash",
        "sameFrameOfReference",
        "sagittalRepresentativeGeometryComplete",
        "axialRepresentativeGeometryComplete",
        "geometryPairingCandidate",
        "fullSeriesGeometryValidated",
        "automaticCrossReferenceValidated",
    ],
)

if geometry_pairing.empty:
    geometry_candidate_count = 0
else:
    geometry_candidate_count = int(
        geometry_pairing[
            "geometryPairingCandidate"
        ].sum()
    )

print(
    "Pares sagital↔axial candidatos por header representativo "
    "dentro de Al-Kafri:",
    geometry_candidate_count,
)

display(geometry_pairing.head(50))

Pares sagital↔axial candidatos por header representativo dentro de Al-Kafri: 0


,sourceFamily,patientHash,studyUidHash,sagittalSeriesUidHash,axialSeriesUidHash,sameFrameOfReference,sagittalRepresentativeGeometryComplete,axialRepresentativeGeometryComplete,geometryPairingCandidate,fullSeriesGeometryValidated,automaticCrossReferenceValidated
0,ALKAFRI,NaN,016ca613229274a6241f2624f31ca8b636413c1d968c0e...,8203155db515e1f31209252b9cac8001f2939856ae9d44...,ce9db28ab7347ce4015e27fcd1cc0fb59fa056d41f977f...,False,True,True,False,False,False
1,ALKAFRI,NaN,016ca613229274a6241f2624f31ca8b636413c1d968c0e...,8203155db515e1f31209252b9cac8001f2939856ae9d44...,d0c6e52818850efc2e055ac20479b2f353409deb61858f...,False,True,True,False,False,False
2,ALKAFRI,NaN,016ca613229274a6241f2624f31ca8b636413c1d968c0e...,a08d1b958c5aa8a586a03d0f0af9f652527046edbcddf5...,ce9db28ab7347ce4015e27fcd1cc0fb59fa056d41f977f...,False,True,True,False,False,False
3,ALKAFRI,NaN,016ca613229274a6241f2624f31ca8b636413c1d968c0e...,a08d1b958c5aa8a586a03d0f0af9f652527046edbcddf5...,d0c6e52818850efc2e055ac20479b2f353409deb61858f...,False,True,True,False,False,False
4,ALKAFRI,NaN,016ca613229274a6241f2624f31ca8b636413c1d968c0e...,a0d79c4a2ce193c1a43f2085a6baf2950e70b6a7bf5366...,ce9db28ab7347ce4015e27fcd1cc0fb59fa056d41f977f...,False,True,True,False,False,False
5,ALKAFRI,NaN,016ca613229274a6241f2624f31ca8b636413c1d968c0e...,a0d79c4a2ce193c1a43f2085a6baf2950e70b6a7bf5366...,d0c6e52818850efc2e055ac20479b2f353409deb61858f...,False,True,True,False,False,False
6,ALKAFRI,NaN,021be4ac6a109be35a0a5e4ee7281fa1464d4aa2bf9dde...,25b7cefb4772fed2788353edccacb26eac7e304e102ffe...,1bb20d56e100a6ae37ab95333fc2fc8233b998086d7582...,False,True,True,False,False,False
7,ALKAFRI,NaN,021be4ac6a109be35a0a5e4ee7281fa1464d4aa2bf9dde...,25b7cefb4772fed2788353edccacb26eac7e304e102ffe...,b180b8f9f66c0cbb6fc9b244e8b9f736ef93e190f49efe...,False,True,True,False,False,False
8,ALKAFRI,NaN,021be4ac6a109be35a0a5e4ee7281fa1464d4aa2bf9dde...,493b5190f9dc7625ced31ba50c65a333e147cfc579b276...,1bb20d56e100a6ae37ab95333fc2fc8233b998086d7582...,False,True,True,False,False,False
9,ALKAFRI,NaN,021be4ac6a109be35a0a5e4ee7281fa1464d4aa2bf9dde...,493b5190f9dc7625ced31ba50c65a333e147cfc579b276...,b180b8f9f66c0cbb6fc9b244e8b9f736ef93e190f49efe...,False,True,True,False,False,False


In [12]:
# Evaluación consolidada: nunca mezcla cohortes.
alk_t1_t2_pairs = int(
    alkafri_study_readiness[
        "hasBothT1T2"
    ].sum()
)
alk_cross_plane_studies = int(
    alkafri_study_readiness[
        "hasBothSagittalAxial"
    ].sum()
)

spider_t1_t2_available = bool(
    spider_summary["rowsWithBothT1T2"] > 0
)

t1_t2_audit = pd.DataFrame(
    [
        {
            "sourceFamily": "ALKAFRI",
            "sameCohortT1T2Evidence":
                alk_t1_t2_pairs > 0,
            "pairedUnit":
                "patientHash+studyUidHash",
            "pairedUnitCount":
                alk_t1_t2_pairs,
            "geometryMetadataAudited": True,
            "crossPlaneGeometryCandidateCount":
                geometry_candidate_count,
            "automaticCrossPlaneAlignmentValidated":
                False,
        },
        {
            "sourceFamily": "P10_7_SPIDER",
            "sameCohortT1T2Evidence":
                spider_t1_t2_available,
            "pairedUnit":
                "manifest_row_patient_level_context",
            "pairedUnitCount":
                int(
                    spider_summary[
                        "rowsWithBothT1T2"
                    ]
                ),
            "geometryMetadataAudited": False,
            "crossPlaneGeometryCandidateCount":
                0,
            "automaticCrossPlaneAlignmentValidated":
                False,
        },
    ]
)

display(t1_t2_audit)


,sourceFamily,sameCohortT1T2Evidence,pairedUnit,pairedUnitCount,geometryMetadataAudited,crossPlaneGeometryCandidateCount,automaticCrossPlaneAlignmentValidated
0,ALKAFRI,True,patientHash+studyUidHash,195,True,0,False
1,P10_7_SPIDER,True,manifest_row_patient_level_context,1518,False,0,False


In [13]:
alignment_gate = {
    "schemaVersion":
        "pfi.p10-8.t1-t2-sagittal-axial-alignment.v1",
    "trainingExecuted": False,
    "weightsDeserialized": False,
    "trainingAuthorized": False,
    "clinicalGroundTruthCreated": False,
    "pixelDataRead": False,
    "patientIdentifiersExported": False,
    "dicomUidsExportedInClear": False,
    "crossCohortPairingAllowed": False,
    "sameCohortT1T2Evidence": {
        "ALKAFRI":
            bool(alk_t1_t2_pairs > 0),
        "P10_7_SPIDER":
            spider_t1_t2_available,
    },
    "samePatientSameStudySagittalAxialEvidence": {
        "ALKAFRI":
            bool(alk_cross_plane_studies > 0),
        "P10_7_SPIDER":
            False,
    },
    "dicomGeometryCrossPlaneCandidates": {
        "ALKAFRI":
            geometry_candidate_count,
        "P10_7_SPIDER":
            0,
    },
    "automaticSagittalAxialAlignmentValidated":
        False,
    "fullProductMultiplanarValidation":
        False,
    "notClinicalDiagnosis": True,
    "professionalReviewRequired": True,
    "trainingPrerequisitesMet": False,
    "reasonTrainingBlocked":
        "No candidate P10.8 finding passed the "
        "viability gate; cross-plane automatic "
        "alignment is not validated.",
}

print(
    json.dumps(
        alignment_gate,
        indent=2,
        ensure_ascii=False,
    )
)


{
  "schemaVersion": "pfi.p10-8.t1-t2-sagittal-axial-alignment.v1",
  "trainingExecuted": false,
  "weightsDeserialized": false,
  "trainingAuthorized": false,
  "clinicalGroundTruthCreated": false,
  "pixelDataRead": false,
  "patientIdentifiersExported": false,
  "dicomUidsExportedInClear": false,
  "crossCohortPairingAllowed": false,
  "sameCohortT1T2Evidence": {
    "ALKAFRI": true,
    "P10_7_SPIDER": true
  },
  "samePatientSameStudySagittalAxialEvidence": {
    "ALKAFRI": true,
    "P10_7_SPIDER": false
  },
  "dicomGeometryCrossPlaneCandidates": {
    "ALKAFRI": 0,
    "P10_7_SPIDER": 0
  },
  "automaticSagittalAxialAlignmentValidated": false,
  "fullProductMultiplanarValidation": false,
  "notClinicalDiagnosis": true,
  "professionalReviewRequired": true,
  "trainingPrerequisitesMet": false,
  "reasonTrainingBlocked": "No candidate P10.8 finding passed the viability gate; cross-plane automatic alignment is not validated."
}


In [14]:
OUT.mkdir(parents=True, exist_ok=True)

output_paths = {
    "dicomHeaderAudit":
        OUT / "alkafri_dicom_header_audit_v1.csv",
    "seriesRegistry":
        OUT / "alkafri_series_registry_v1.csv",
    "alkafriStudyReadiness":
        OUT / "alkafri_study_t1_t2_plane_readiness_v1.csv",
    "spiderT1T2Summary":
        OUT / "spider_t1_t2_readiness_v1.json",
    "geometryPairing":
        OUT / "same_cohort_geometry_pairing_candidates_v1.csv",
    "t1T2Audit":
        OUT / "same_cohort_t1_t2_audit_v1.csv",
    "alignmentGate":
        OUT / "t1_t2_sagittal_axial_alignment_gate_v1.json",
    "inputHashes":
        OUT / "notebook75_input_hashes_v1.csv",
    "summary":
        OUT / "NOTEBOOK_76_SUMMARY.json",
}

# Ningún PatientID ni UID DICOM en claro se exporta.
dicom_headers.to_csv(
    output_paths["dicomHeaderAudit"],
    index=False,
)
series_registry.to_csv(
    output_paths["seriesRegistry"],
    index=False,
)
alkafri_study_readiness.to_csv(
    output_paths["alkafriStudyReadiness"],
    index=False,
)
write_json(
    output_paths["spiderT1T2Summary"],
    spider_summary,
)
geometry_pairing.to_csv(
    output_paths["geometryPairing"],
    index=False,
)
t1_t2_audit.to_csv(
    output_paths["t1T2Audit"],
    index=False,
)
write_json(
    output_paths["alignmentGate"],
    alignment_gate,
)
pd.DataFrame(
    [
        {"inputName": k, "sha256": v}
        for k, v in input_hashes.items()
    ]
).to_csv(
    output_paths["inputHashes"],
    index=False,
)

pt_files = list(OUT.rglob("*.pt"))
if pt_files:
    raise RuntimeError(
        "La salida Notebook 76 contiene .pt inesperados."
    )

summary = {
    "schemaVersion":
        "pfi.p10-8.notebook-76-summary.v1",
    "generatedAtUtc":
        datetime.now(timezone.utc).isoformat(),
    "trainingExecuted": False,
    "weightsDeserialized": False,
    "trainingAuthorized": False,
    "internalTestAccessed": False,
    "officialHiddenTestAccessed": False,
    "pixelDataRead": False,
    "patientIdentifiersExported": False,
    "dicomUidsExportedInClear": False,
    "clinicalGroundTruthCreated": False,
    "crossCohortPairingAllowed": False,
    "notClinicalDiagnosis": True,
    "professionalReviewRequired": True,
    "dicomHeaderCount":
        int(len(dicom_headers)),
    "dicomReadSuccessCount":
        int(
            dicom_headers[
                "readSucceeded"
            ].sum()
        ),
    "alkafriSeriesCount":
        int(len(series_registry)),
    "alkafriStudyCount":
        int(len(alkafri_study_readiness)),
    "alkafriStudiesWithBothT1T2":
        alk_t1_t2_pairs,
    "alkafriStudiesWithBothSagittalAxial":
        alk_cross_plane_studies,
    "alkafriGeometryPairingCandidateCount":
        geometry_candidate_count,
    "spiderRowsWithBothT1T2":
        int(
            spider_summary[
                "rowsWithBothT1T2"
            ]
        ),
    "automaticSagittalAxialAlignmentValidated":
        False,
    "fullProductMultiplanarValidation":
        False,
    "trainingPrerequisitesMet":
        False,
    "outputPtFileCount": len(pt_files),
    "nextRequiredGate":
        "P10_8_PREFLIGHT_CLOSE_OR_NEW_DATASET_DECISION",
}

write_json(
    output_paths["summary"],
    summary,
)

marker = {
    **summary,
    "schemaVersion":
        "pfi.p10-8.notebook-76-complete.v1",
    "summarySchemaVersion":
        summary["schemaVersion"],
    "status": "NOTEBOOK_76_COMPLETE",
    "outputs": {
        key: str(value)
        for key, value in output_paths.items()
    },
}

write_json(
    OUT / "NOTEBOOK_76_COMPLETE.json",
    marker,
)

print(
    json.dumps(
        marker,
        indent=2,
        ensure_ascii=False,
    )
)
print("NOTEBOOK_76_COMPLETE")


{
  "schemaVersion": "pfi.p10-8.notebook-76-complete.v1",
  "generatedAtUtc": "2026-08-07T03:27:15.343303+00:00",
  "trainingExecuted": false,
  "weightsDeserialized": false,
  "trainingAuthorized": false,
  "internalTestAccessed": false,
  "officialHiddenTestAccessed": false,
  "pixelDataRead": false,
  "patientIdentifiersExported": false,
  "dicomUidsExportedInClear": false,
  "clinicalGroundTruthCreated": false,
  "crossCohortPairingAllowed": false,
  "notClinicalDiagnosis": true,
  "professionalReviewRequired": true,
  "dicomHeaderCount": 1363,
  "dicomReadSuccessCount": 1363,
  "alkafriSeriesCount": 1363,
  "alkafriStudyCount": 204,
  "alkafriStudiesWithBothT1T2": 195,
  "alkafriStudiesWithBothSagittalAxial": 198,
  "alkafriGeometryPairingCandidateCount": 0,
  "spiderRowsWithBothT1T2": 1518,
  "automaticSagittalAxialAlignmentValidated": false,
  "fullProductMultiplanarValidation": false,
  "trainingPrerequisitesMet": false,
  "outputPtFileCount": 0,
  "nextRequiredGate": "P10_8_PR

## Interpretación obligatoria

`NOTEBOOK_76_COMPLETE` significa que se auditó la disponibilidad T1/T2 y la
información geométrica necesaria para una futura alineación multiplanar.

No significa que la correspondencia sagital–axial automática esté validada.

Mientras `automaticSagittalAxialAlignmentValidated=false`:

- la asociación entre planos debe ser revisada por un profesional;
- no se deben mezclar cohortes diferentes;
- no se debe usar coincidencia de IDs como sustituto de geometría DICOM;
- no se habilita entrenamiento P10.8;
- cualquier Notebook 77 de entrenamiento requeriría una nueva decisión explícita,
  evidencia viable y aprobación de cátedra.
